##### CAN ID Unique Values

Characterise the set of CAN IDs present in benign traffic and how often each one appears. 

- **ID vocabulary & frequency** — number of unique IDs, most/least frequent IDs, overall frequency distribution.
- **Vocabulary consistency across files / days** — confirm that the same IDs appear in every file/day, i.e. the benign ID vocabulary is stable. 

In [ ]:
import re 
from pathlib import Path 
from collections import Counter 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

In [ ]:
path = Path("/Users/anita/Documents/TFM/SSL_CyberSecurity")
benign_folders = sorted(path.glob("Benign*"))
pattern = re.compile(
    r"\((.*?)\)\s+(\S+)\s+([0-9A-Fa-f]+)#([0-9A-Fa-f]*)\s*(\d*)"
)

In [ ]:
# Count how many benign files are in the benign folders
benign_files = sorted(
    log_file
    for folder in benign_folders 
    for log_file in folder.rglob("*.log")
)
print(f"Benign files found: {len(benign_files)}")

In [ ]:
def extract_can_id(log_file): 
    can_ids = []
    
    with open(log_file) as f: 
        for line in f: 
            match = pattern.match(line)
            if match:
                can_ids.append(match.group(3))
    return can_ids 

In [ ]:
benign_files = []

for folder in benign_folders:
    for path in folder.rglob("*.log"): 
        benign_files.append(path)
    
benign_files = sorted(benign_files)

benign_can_ids = Counter()

for path in benign_files: 
    benign_can_ids.update(extract_can_id(path))
       
print(f"Total unique CAN IDs: {len(benign_can_ids)}")
print(f"Benign files scanned: {len(benign_files)}")

print("\nTop 10 most common CAN IDs: ")

for can_id, count in benign_can_ids.most_common(10): 
    print(f"{can_id}: {count}")

Scanning all 12 benign log files we can identify 56 unique CAN IDs, consistent with the number of IDs that the oficial CAN-MIRGU paper states. This indicates that the complete normal CAN ID vocabulary was succesfully extracted. 


In [ ]:
id_freq_df = pd.DataFrame(
    benign_can_ids.items(),
    columns=["can_id", "count"]
)

id_freq_df = id_freq_df.sort_values("count", ascending=False)

id_freq_df = id_freq_df.reset_index(drop=True)

print(id_freq_df.head())

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(
    id_freq_df["can_id"], 
    id_freq_df["count"], 
)

plt.xlabel("CAN ID")
plt.ylabel("Frequency")
plt.title("Frequency of CAN IDs in Benign Log Files")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()